# Lia — Qwen Base com clonagem ICL e referências de emoção (T4)

**Abordagem principal: preservar a clonagem que já funcionava.** Usa o Qwen3-TTS Base congelado com `ref_audio` **e a transcrição exata `ref_text`** (modo ICL completo). Não usa LoRA, speaker embedding isolado (`x_vector_only_mode`), nem fine-tuning. A saída deve ser avaliada como amostra experimental, não como voz final.

O segundo eixo modifica a prosódia pela escolha de uma fala de referência da própria dubladora marcada/revisada como alegre, neutra ou outra emoção. O Qwen Base não oferece argumento `instruct` na API de clonagem; não vamos fingir que um prompt textual controla a emoção. A hipótese testada é transferência de prosódia via referência ICL, mantendo o timbre da dubladora.

Usa o pacote curado já exportado (ou os arquivos existentes em `/content/lia_qwen_lora`). Não pede o WAV original e não treina. O controle de identidade é a referência exata que já funcionou no teste Base. Áudios e logs ficam no runtime privado Colab; sem serviços hosted.


In [ ]:
# 0. Reutilizar pacote curado/runtime; não pede o WAV original
from pathlib import Path
import os, sys, subprocess, json, csv, hashlib, time, zipfile, shutil
ROOT=Path('/content/lia_qwen_lora'); ROOT.mkdir(parents=True,exist_ok=True)
required=['speaker_reference.wav','review.csv']
def bundle_ready(): return all((ROOT/x).is_file() for x in required) and any((ROOT/x).is_file() for x in ['train_t4.jsonl','train_raw.jsonl'])
if not bundle_ready():
    zips=sorted(ROOT.glob('*.zip'))
    if len(zips)==1: bundle=zips[0]
    else:
        from google.colab import files
        print('Envie somente o ZIP CURADO de treino previamente baixado (não o WAV original).')
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError('Envie exatamente um ZIP curado.')
        name,data=next(iter(uploaded.items()))
        if not name.lower().endswith('.zip'): raise ValueError('Esperado ZIP curado.')
        bundle=ROOT/Path(name).name; bundle.write_bytes(data); del data,uploaded
    with zipfile.ZipFile(bundle) as zf:
        root=ROOT.resolve()
        for item in zf.infolist():
            target=(ROOT/item.filename).resolve()
            if target!=root and root not in target.parents: raise ValueError('Caminho inseguro no ZIP; extração bloqueada.')
        zf.extractall(ROOT)
if not bundle_ready(): raise FileNotFoundError(f'Pacote sem referência/revisão/JSONL curado em {ROOT}')
approval_path=ROOT/'review_approval.json'; manifest_path=ROOT/'training_bundle_manifest.json'
if approval_path.is_file() and manifest_path.is_file():
    approval=json.loads(approval_path.read_text(encoding='utf-8')); bundle_manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
    review_hash=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest()
    if approval.get('review_csv_sha256')!=review_hash or bundle_manifest.get('review_csv_sha256')!=review_hash:
        raise RuntimeError('Hash da revisão difere da aprovação/pacote; não usar referências até reconciliar o ZIP.')
print('Reutilizando os arquivos privados em',ROOT)
# Preferir o split T4 já curado; senão usar o split de treino do bundle.
TRAIN_JSONL=ROOT/('train_t4.jsonl' if (ROOT/'train_t4.jsonl').is_file() else 'train_raw.jsonl')


## 1. Selecionar referências ICL pela curadoria humana

A amostra-base usa `speaker_reference.wav` e a transcrição marcada como referência no `review.csv`. Para emoção, selecionamos no máximo uma fala de treino por classe, somente entre os exemplos curados; holdout não é usado como prompt. Se não houver um exemplo de uma classe, ela é omitida. Nenhum ASR ou classificador automático decide a emoção nesta etapa.


In [ ]:
# Exigir uma transcrição exata para a referência que produziu a voz-base conhecida
with (ROOT/'review.csv').open(encoding='utf-8-sig',newline='') as f: reviews=list(csv.DictReader(f))
refs=[r for r in reviews if r.get('reference','').strip().lower() in {'yes','true','1','sim'}]
if len(refs)!=1: raise RuntimeError(f'Esperava exatamente uma referência aprovada em review.csv; achei {len(refs)}.')
base_ref=ROOT/'speaker_reference.wav'; base_text=refs[0].get('text','').strip()
if not base_text: raise ValueError('A transcrição exata da referência está vazia; corrigir no review.csv antes de sintetizar.')

def resolve_audio(value):
    p=Path(value)
    if p.is_file() and p.resolve().is_relative_to(ROOT.resolve()): return p.resolve()
    matches=[x.resolve() for x in ROOT.rglob(p.name) if x.is_file() and x.suffix.lower()=='.wav']
    if len(matches)!=1: raise FileNotFoundError(f'Áudio {value!r} não resolvido univocamente: {matches[:5]}')
    return matches[0]
rows=[json.loads(line) for line in TRAIN_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
import soundfile as sf
from IPython.display import Audio, Markdown, display
labels=['alegre','comemorativa','neutro','calma','assertiva','triste','irritada']
# Opcional: depois de ouvir as prévias abaixo, defina {'alegre':'segment_id', ...}
# usando o id do recorte exibido. Vazio = melhor duração, NÃO significa maior intensidade.
REFERENCE_ID_OVERRIDE={}
PREVIEW_PER_EMOTION=3
cases=[{'label':'base_icl_reference','ref_audio':str(base_ref),'ref_text':base_text,'emotion':'base'}]
candidate_bank={}
for emotion in labels:
    candidates=[]
    for row in rows:
        if str(row.get('emotion_audit','')).strip().lower()!=emotion: continue
        try: audio=resolve_audio(row['audio'])
        except (KeyError,FileNotFoundError): continue
        text=str(row.get('text','')).strip()
        if text:
            duration=float(sf.info(audio).duration)
            if 3.0 <= duration <= 10.0:
                candidates.append({'id':audio.stem,'audio':audio,'text':text,'seconds':duration})
    candidates.sort(key=lambda item:(item['seconds'],item['id']),reverse=True)
    if candidates:
        candidate_bank[emotion]=candidates
        chosen_id=REFERENCE_ID_OVERRIDE.get(emotion)
        chosen=next((item for item in candidates if item['id']==chosen_id),None) if chosen_id else candidates[0]
        if chosen is None: raise ValueError(f'Override {emotion}={chosen_id!r} não está entre os recortes elegíveis.')
        cases.append({'label':f'ref_{emotion}','ref_audio':str(chosen['audio']),'ref_text':chosen['text'],'emotion':emotion,'reference_seconds':chosen['seconds'],'reference_id':chosen['id']})
print('Controle ICL:',base_ref.name,'| transcrição:',base_text)
for emotion,candidates in candidate_bank.items():
    display(Markdown(f"### Referências candidatas: {emotion} — ouça e escolha a mais expressiva\nAs prévias são ordenadas por duração, não por intensidade emocional. Para trocar a escolhida, defina `REFERENCE_ID_OVERRIDE['{emotion}']` com um dos IDs abaixo e execute novamente esta célula."))
    for item in candidates[:PREVIEW_PER_EMOTION]:
        display(Markdown(f"**ID `{item['id']}` · {item['seconds']:.1f} s** — {item['text']}"))
        display(Audio(filename=str(item['audio'])))
print('Referências selecionadas para síntese:')
for item in cases: print(item['label'],'| id=',item.get('reference_id','base'),'|',Path(item['ref_audio']).name,'|',item['ref_text'])
if len(cases)==1: print('Aviso: nenhum recorte emocional elegível no split de treino; só será gerado o controle base.')


## 2. Carregar Qwen Base e gerar A/B sem alterar pesos

A inferência usa o mesmo modo de clonagem do teste que funcionou: referência + transcrição. A linguagem é explicitamente `Portuguese`, como na avaliação Base anterior; isso não prova PT-BR — a regionalidade precisa ser ouvida. Executa apenas inferência, sem treino, instalação global ou envio do áudio.


In [ ]:
# Inferência Base em processo isolado: FP32 + SDPA para T4; modelo fixado
QWEN_COMMIT='0c6a7cbb6c8421a46332f8c2434c7825c4c855ef'
MODEL_ID='Qwen/Qwen3-TTS-12Hz-0.6B-Base'; MODEL_REV='5d83992436eae1d760afd27aff78a71d676296fc'
DEPS=ROOT/'python_deps'; QWEN_DIR=ROOT/'Qwen3-TTS-base-icl'; DEPS.mkdir(exist_ok=True)
packages=['transformers==4.57.3','accelerate==1.12.0','huggingface_hub==0.36.2','safetensors==0.7.0','tokenizers==0.22.1','librosa==0.11.0','soxr==0.5.0.post1','sox==1.5.0','einops==0.8.1','onnxruntime==1.23.2']
marker=DEPS/'.icl_deps_ready'
if not marker.exists():
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),*packages],check=True)
    marker.write_text('\n'.join(packages)+'\n',encoding='utf-8')
if not QWEN_DIR.exists():
    subprocess.run(['git','clone','--quiet','--no-checkout','https://github.com/QwenLM/Qwen3-TTS.git',str(QWEN_DIR)],check=True)
subprocess.run(['git','-C',str(QWEN_DIR),'fetch','--quiet','--depth','1','origin',QWEN_COMMIT],check=True)
subprocess.run(['git','-C',str(QWEN_DIR),'checkout','--force',QWEN_COMMIT],check=True)
model_path=''
from huggingface_hub import snapshot_download
os.environ['HF_HOME']=str(ROOT/'hf_home')
model_path=snapshot_download(MODEL_ID,revision=MODEL_REV,cache_dir=str(ROOT/'hf_home'))
if not __import__('torch').cuda.is_available(): raise RuntimeError('Ative uma GPU CUDA (T4) antes da inferência.')
print('Qwen Base snapshot:',model_path,'| Qwen commit:',QWEN_COMMIT)


## 3. Gerar e ouvir o conjunto de controle

Mesmo texto, mesma seed e mesmo Qwen Base; varia apenas o áudio/transcrição de referência. O controle `base_icl_reference` é o caminho que preserva a clonagem ICL. As outras amostras testam se referências com emoção revisada transferem prosódia sem perder a identidade. Não há argumento textual `instruct` no método Base `generate_voice_clone`; não passamos um parâmetro inexistente.


In [ ]:
from IPython.display import Audio, Markdown, display
import soundfile as sf
TARGET_TEXT='Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
run_tag=time.strftime('%Y%m%d_%H%M%S')
manifest={'approach':'frozen_base_full_icl_reference_transfer','model_revision':MODEL_REV,'qwen_commit':QWEN_COMMIT,'language':'Portuguese','target_text':TARGET_TEXT,'cases':[]}
program=r'''import json,sys,hashlib
from pathlib import Path
import soundfile as sf
import torch
sys.path.insert(0,sys.argv[1]);sys.path.insert(0,sys.argv[2])
import transformers.utils.import_utils as iu
 iu._torchvision_available=False
from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel
root=Path(sys.argv[3]); model_id=sys.argv[4]; rev=sys.argv[5]; outdir=Path(sys.argv[6]); target=sys.argv[7]
cases=json.loads(Path(sys.argv[8]).read_text(encoding='utf-8'))
model_dir=snapshot_download(model_id,revision=rev,cache_dir=str(root/'hf_home'))
model=Qwen3TTSModel.from_pretrained(model_dir,device_map='cuda:0',dtype=torch.float32,attn_implementation='sdpa')
for item in cases:
    print('Gerando',item['label'],'com ICL full; ref=',item['ref_audio'],flush=True)
    torch.manual_seed(1234)
    prompt=model.create_voice_clone_prompt(ref_audio=item['ref_audio'],ref_text=item['ref_text'],x_vector_only_mode=False)
    wavs,sr=model.generate_voice_clone(text=target,language='Portuguese',voice_clone_prompt=prompt,max_new_tokens=160,do_sample=True)
    path=outdir/f"lia_{item['label']}_{sys.argv[9]}.wav"
    sf.write(path,wavs[0],sr)
    print(json.dumps({'label':item['label'],'path':str(path),'seconds':len(wavs[0])/sr,'sample_rate':sr,'ref_sha256':hashlib.sha256(Path(item['ref_audio']).read_bytes()).hexdigest()},ensure_ascii=False),flush=True)
'''
# Corrige indentação do corpo Python embutido.
import textwrap
program=textwrap.dedent(program).replace('\n iu._torchvision_available=False','\niu._torchvision_available=False')
case_path=ROOT/f'icl_cases_{run_tag}.json'; case_path.write_text(json.dumps(cases,ensure_ascii=False),encoding='utf-8')
script_path=ROOT/'run_base_icl.py'; script_path.write_text(program,encoding='utf-8')
outdir=ROOT/'base_icl_outputs'; outdir.mkdir(exist_ok=True)
env=os.environ.copy(); env['HF_HOME']=str(ROOT/'hf_home'); env['PYTHONUNBUFFERED']='1'; env['USE_TF']='0'; env['USE_FLAX']='0'
env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(QWEN_DIR),env.get('PYTHONPATH','')])
cmd=[sys.executable,'-u',str(script_path),str(DEPS),str(QWEN_DIR),str(ROOT),MODEL_ID,MODEL_REV,str(outdir),TARGET_TEXT,str(case_path),run_tag]
proc=subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait()!=0: raise RuntimeError('Inferência Base ICL falhou; consulte o traceback/log acima.')
outputs=sorted(outdir.glob(f'*_{run_tag}.wav'))
if not outputs: raise RuntimeError('Nenhum WAV novo; recusando reproduzir saída antiga.')
display(Markdown('## Áudios gerados — cada player está identificado pelo rótulo e pelo arquivo'))
titles={'base_icl_reference':'CONTROLE — Qwen Base com a referência ICL que já funcionava',
        'ref_alegre':'REFERÊNCIA EMOCIONAL — alegre', 'ref_comemorativa':'REFERÊNCIA EMOCIONAL — comemorativa',
        'ref_neutro':'REFERÊNCIA EMOCIONAL — neutra', 'ref_calma':'REFERÊNCIA EMOCIONAL — calma',
        'ref_assertiva':'REFERÊNCIA EMOCIONAL — assertiva', 'ref_triste':'REFERÊNCIA EMOCIONAL — triste',
        'ref_irritada':'REFERÊNCIA EMOCIONAL — irritada'}
label_by_file={f"lia_{item['label']}_{run_tag}.wav":item['label'] for item in cases}
for path in outputs:
    label=label_by_file.get(path.name,path.stem)
    display(Markdown(f"### {titles.get(label,label)}\n\nArquivo: `{path.name}`"))
    display(Audio(filename=str(path)))
manifest['outputs']=[{'path':str(p),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in outputs]
manifest_path=ROOT/f'base_icl_manifest_{run_tag}.json'; manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Manifesto:',manifest_path)


### Como interpretar
- Se `base_icl_reference` voltar ao timbre que você já reconhecia, isso confirma que a regressão veio da rota LoRA/custom-voice, não da falta de adaptação do modelo-base.
- Compare as variantes emocionais ouvindo timbre e prosódia. A hipótese é transferência de estilo por referência; se não variar emoção o bastante, não alegaremos controle emocional resolvido.
- As etiquetas servem só para escolher referências humanas revisadas. Não são condicionamento supervisionado nem garantia de emoção.
- O próximo passo, se este teste confirmar a voz-base, é manter esse caminho ICL na Lia e integrar a seleção de referência como controle de estilo. Treino de pesos só deve voltar se houver uma técnica que preserve o prompt ICL e seja avaliada contra este controle.
